# KMeans clustering on case_day Dice + submit sample export

This notebook clusters `case_day` entries by `dice_mean` (from `eval_per_case.csv`) using KMeans, then exports **one representative sample** from the **best** and **worst** clusters as two smaller `submit.csv` files (subset of rows filtered by `case_day`).

Inputs:
- `eval_per_case.csv`: must contain `case_day` and `dice_mean` (optionally per-class dice).
- `submit.csv`: Kaggle-style RLE submission (`id,class,segmentation`).

Outputs (written to `out_dir`):
- `clusters.csv`, `cluster_stats.csv`, `report.txt`
- `best_cluster_sample__<case_day>__submit.csv`
- `worst_cluster_sample__<case_day>__submit.csv`


In [1]:
from __future__ import annotations

import csv
from pathlib import Path
import os
import sys

def find_repo_root(start: Path) -> Path:
    for p in (start, *start.parents):
        if (p / "inputs" / "train.csv").exists():
            return p
        if (p / ".git").exists() and (p / "README.md").exists():
            return p
    raise RuntimeError(
        "Cannot find repo root. Start Jupyter from the project directory, or ensure inputs/train.csv exists."
    )


REPO_ROOT = find_repo_root(Path.cwd().resolve())
os.chdir(REPO_ROOT)
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

print("repo_root:", REPO_ROOT)


import numpy as np
import pandas as pd
import cv2
import matplotlib.pyplot as plt

from sklearn.cluster import KMeans

from src.constants import CLASSES
from src.data_utils import build_case_day_slices, parse_scan_filename
from src.rle import rle_decode

repo_root: /home/ubuntu/UW-Madison_GI_Tract_Image_Segmentation


## Parameters

Set these paths to point at your model's infer outputs.

In [2]:
# Example run directory (edit as needed)
run_dir = Path('outputs/20260301-002909_Unet3D_IMREAD_UNCHANGED/infer')

eval_per_case = run_dir / 'eval_per_case.csv'
submit_csv = run_dir / 'submit.csv'

# Clustering
k = 3
seed = 0

# If True: cluster on [dice_mean + per-class dice]
use_per_class = False

# Output
out_dir = run_dir / 'kmeans_dice_clusters'
out_dir.mkdir(parents=True, exist_ok=True)

eval_per_case, submit_csv, out_dir

(PosixPath('outputs/20260301-002909_Unet3D_IMREAD_UNCHANGED/infer/eval_per_case.csv'),
 PosixPath('outputs/20260301-002909_Unet3D_IMREAD_UNCHANGED/infer/submit.csv'),
 PosixPath('outputs/20260301-002909_Unet3D_IMREAD_UNCHANGED/infer/kmeans_dice_clusters'))

## Load eval + build features

In [3]:
df = pd.read_csv(eval_per_case)
if 'case_day' not in df.columns or 'dice_mean' not in df.columns:
    raise ValueError('eval_per_case.csv must contain columns: case_day, dice_mean')

df = df.copy()
df['dice_mean'] = pd.to_numeric(df['dice_mean'], errors='coerce')
df = df[np.isfinite(df['dice_mean'].to_numpy(dtype=float, copy=False))].reset_index(drop=True)
if len(df) == 0:
    raise ValueError('No finite dice_mean rows found')

feat_cols = ['dice_mean']
if use_per_class:
    need = ['dice_large_bowel', 'dice_small_bowel', 'dice_stomach']
    missing = [c for c in need if c not in df.columns]
    if missing:
        raise ValueError(f'Missing per-class dice columns: {missing}')
    for c in need:
        df[c] = pd.to_numeric(df[c], errors='coerce')
    df = df.dropna(subset=need).reset_index(drop=True)
    feat_cols = ['dice_mean', *need]

X = df[feat_cols].to_numpy(dtype=np.float64, copy=True)
df[['case_day', 'dice_mean']].head()

,case_day,dice_mean
0,case101_day20,0.825134
1,case101_day22,0.847434
2,case101_day26,0.868241
3,case101_day32,0.865923
4,case102_day0,0.820469


## KMeans clustering

In [4]:
if len(df) < k:
    raise ValueError(f'Need at least k samples (k={k}) but got {len(df)}')

km = KMeans(n_clusters=int(k), random_state=int(seed), n_init='auto')
labels = km.fit_predict(X).astype(int)
centers = km.cluster_centers_

df['cluster'] = labels

cluster_stats = (
    df.groupby('cluster', as_index=False)
      .agg(
          n=('case_day', 'size'),
          dice_mean_avg=('dice_mean', 'mean'),
          dice_mean_min=('dice_mean', 'min'),
          dice_mean_max=('dice_mean', 'max'),
      )
      .sort_values('dice_mean_avg', ascending=True)
      .reset_index(drop=True)
)
cluster_stats['rank'] = np.arange(len(cluster_stats), dtype=int)
cluster_stats

,cluster,n,dice_mean_avg,dice_mean_min,dice_mean_max,rank
0,2,1,0.498724,0.498724,0.498724,0
1,0,106,0.850690,0.789932,0.870643,1
2,1,109,0.890662,0.871686,0.923385,2


## Pick representative case_day from best / worst cluster

We pick the **medoid** (closest to the cluster center in feature space).

In [5]:
worst_cluster = int(cluster_stats.iloc[0]['cluster'])
best_cluster = int(cluster_stats.iloc[-1]['cluster'])

picked = {}
for which, cl in [('worst', worst_cluster), ('best', best_cluster)]:
    sub = df[df['cluster'] == cl].copy().reset_index(drop=True)
    subX = sub[feat_cols].to_numpy(dtype=np.float64, copy=True)
    c = centers[cl].reshape(1, -1)
    d = np.sqrt(((subX - c) ** 2).sum(axis=1))
    picked[which] = str(sub.iloc[int(d.argmin())]['case_day'])

best_case_day = picked['best']
worst_case_day = picked['worst']
best_cluster, best_case_day, worst_cluster, worst_case_day

(1, 'case125_day15', 2, 'case81_day30')

## Export submit.csv subsets for those case_days

In [6]:
def write_submit_subset(submit_csv: Path, *, out_csv: Path, case_day: str) -> int:
    out_csv.parent.mkdir(parents=True, exist_ok=True)
    prefix = f"{case_day}_slice_"
    n = 0
    with open(submit_csv, 'r', encoding='utf-8', newline='') as fin, open(out_csv, 'w', encoding='utf-8', newline='') as fout:
        r = csv.DictReader(fin)
        w = csv.DictWriter(fout, fieldnames=['id', 'class', 'segmentation'])
        w.writeheader()
        for row in r:
            _id = str(row.get('id', ''))
            if _id.startswith(prefix):
                w.writerow({'id': row.get('id', ''), 'class': row.get('class', ''), 'segmentation': row.get('segmentation', '')})
                n += 1
    return int(n)

clusters_csv = out_dir / 'clusters.csv'
cluster_stats_csv = out_dir / 'cluster_stats.csv'

# Save full cluster assignment table (join rank for convenience)
df2 = df.merge(cluster_stats[['cluster', 'rank', 'dice_mean_avg']], on='cluster', how='left')
df2.sort_values(['rank', 'dice_mean'], ascending=[True, True]).to_csv(clusters_csv, index=False)
cluster_stats.to_csv(cluster_stats_csv, index=False)

best_out = out_dir / f"best_cluster_sample__{best_case_day}__submit.csv"
worst_out = out_dir / f"worst_cluster_sample__{worst_case_day}__submit.csv"

n_best = write_submit_subset(submit_csv, out_csv=best_out, case_day=best_case_day)
n_worst = write_submit_subset(submit_csv, out_csv=worst_out, case_day=worst_case_day)

report = out_dir / 'report.txt'
lines = []
lines.append(f"eval_per_case: {eval_per_case}")
lines.append(f"submit_csv  : {submit_csv}")
lines.append(f"k={int(k)} seed={int(seed)} features={feat_cols}")
lines.append('')
lines.append('cluster_stats:')
for _, r in cluster_stats.iterrows():
    lines.append(
        f"  rank={int(r['rank'])} cluster={int(r['cluster'])} n={int(r['n'])} "
        f"mean={float(r['dice_mean_avg']):.6f} min={float(r['dice_mean_min']):.6f} max={float(r['dice_mean_max']):.6f}"
    )
lines.append('')
lines.append(f"best_cluster: {best_cluster} -> sample case_day={best_case_day} submit_rows={n_best}")
lines.append(f"worst_cluster: {worst_cluster} -> sample case_day={worst_case_day} submit_rows={n_worst}")
lines.append('')
lines.append(f"clusters_csv: {clusters_csv}")
lines.append(f"best_submit_sample: {best_out}")
lines.append(f"worst_submit_sample: {worst_out}")
report.write_text("\n".join(lines) + "\n", encoding='utf-8')

print("\n".join(lines))

eval_per_case: outputs/20260301-002909_Unet3D_IMREAD_UNCHANGED/infer/eval_per_case.csv
submit_csv  : outputs/20260301-002909_Unet3D_IMREAD_UNCHANGED/infer/submit.csv
k=3 seed=0 features=['dice_mean']

cluster_stats:
  rank=0 cluster=2 n=1 mean=0.498724 min=0.498724 max=0.498724
  rank=1 cluster=0 n=106 mean=0.850690 min=0.789932 max=0.870643
  rank=2 cluster=1 n=109 mean=0.890662 min=0.871686 max=0.923385

best_cluster: 1 -> sample case_day=case125_day15 submit_rows=432
worst_cluster: 2 -> sample case_day=case81_day30 submit_rows=432

clusters_csv: outputs/20260301-002909_Unet3D_IMREAD_UNCHANGED/infer/kmeans_dice_clusters/clusters.csv
best_submit_sample: outputs/20260301-002909_Unet3D_IMREAD_UNCHANGED/infer/kmeans_dice_clusters/best_cluster_sample__case125_day15__submit.csv
worst_submit_sample: outputs/20260301-002909_Unet3D_IMREAD_UNCHANGED/infer/kmeans_dice_clusters/worst_cluster_sample__case81_day30__submit.csv


## Visualization (raw / GT / prediction)

This section picks a slice from each representative `case_day` and plots:
- raw grayscale image
- GT masks (decoded from `inputs/train.csv`)
- predicted masks (decoded from `submit.csv`)
- overlay views


In [7]:
train_csv = Path('inputs/train.csv')

case_day_slices = build_case_day_slices(str(Path('inputs') / 'train'))

def rle_area(seg: str | None) -> int:
    if seg is None:
        return 0
    seg = str(seg).strip()
    if not seg or seg.lower() == 'nan':
        return 0
    parts = seg.split()
    if len(parts) % 2 != 0:
        return 0
    try:
        return int(sum(int(x) for x in parts[1::2]))
    except Exception:
        return 0

def load_rles_for_case_days(csv_path: Path, case_days: set[str]) -> dict[str, dict[int, dict[str, str]]]:
    out: dict[str, dict[int, dict[str, str]]] = {cd: {} for cd in case_days}
    with open(csv_path, 'r', encoding='utf-8', newline='') as f:
        r = csv.DictReader(f)
        for row in r:
            _id = str(row.get('id', '')).strip()
            cls = str(row.get('class', '')).strip()
            seg = str(row.get('segmentation', '') or '').strip()
            if not _id or cls not in CLASSES:
                continue
            parts = _id.split('_')
            if len(parts) < 4:
                continue
            case_day = '_'.join(parts[0:2])
            if case_day not in out:
                continue
            try:
                slice_idx = int(parts[-1])
            except Exception:
                continue
            out[case_day].setdefault(slice_idx, {})[cls] = seg
    return out

def slice_pos_map(case_day: str) -> dict[int, int]:
    m: dict[int, int] = {}
    for pos, p in enumerate(case_day_slices[case_day]):
        _, idx, _, _ = parse_scan_filename(Path(p).name)
        m[int(idx)] = int(pos)
    return m

def pick_slice_idx_by_gt_area(case_day: str, gt_rles: dict[int, dict[str, str]]) -> int:
    if not gt_rles:
        return 1
    best_idx = None
    best_area = -1
    for slice_idx, cls_map in gt_rles.items():
        area = sum(rle_area(cls_map.get(cls, '')) for cls in CLASSES)
        if area > best_area:
            best_area = area
            best_idx = int(slice_idx)
    if best_idx is None:
        return 1
    # If all GT areas are zero, fall back to mid-slice
    if best_area <= 0:
        return int(len(case_day_slices[case_day]) // 2) + 1
    return int(best_idx)

def decode_masks(cls_to_rle: dict[str, str] | None, H: int, W: int) -> dict[str, np.ndarray]:
    cls_to_rle = cls_to_rle or {}
    return {cls: rle_decode(cls_to_rle.get(cls, ''), H, W) for cls in CLASSES}

def plot_case_day_slice(case_day: str, slice_idx: int, *, gt: dict[str, np.ndarray], pred: dict[str, np.ndarray], alpha: float = 0.4):
    pos_map = slice_pos_map(case_day)
    if slice_idx not in pos_map:
        raise KeyError(f'slice_idx={slice_idx} not found for {case_day}')
    slice_pos = pos_map[slice_idx]
    img_path = Path(case_day_slices[case_day][slice_pos])
    img = cv2.imread(str(img_path), cv2.IMREAD_GRAYSCALE)
    if img is None:
        raise FileNotFoundError(f'Failed to read image: {img_path}')

    cmaps = ['Reds', 'Greens', 'Blues']
    fig_grid = plt.figure(figsize=(14, 7))

    ax = plt.subplot(2, 4, 1)
    ax.set_title('image')
    ax.imshow(img, cmap='gray')
    ax.axis('off')

    for i, cls in enumerate(CLASSES):
        ax = plt.subplot(2, 4, 2 + i)
        ax.set_title(f'GT: {cls}')
        ax.imshow(gt[cls], cmap=cmaps[i])
        ax.axis('off')

    ax = plt.subplot(2, 4, 5)
    ax.set_title('image')
    ax.imshow(img, cmap='gray')
    ax.axis('off')

    for i, cls in enumerate(CLASSES):
        ax = plt.subplot(2, 4, 6 + i)
        ax.set_title(f'Pred: {cls}')
        ax.imshow(pred[cls], cmap=cmaps[i])
        ax.axis('off')

    plt.tight_layout()

    fig_overlay = plt.figure(figsize=(12, 6))
    ax1 = plt.subplot(1, 2, 1)
    ax1.set_title('overlay: GT')
    ax1.imshow(img, cmap='gray')
    for i, cls in enumerate(CLASSES):
        ax1.imshow(gt[cls], alpha=float(alpha), cmap=cmaps[i])
    ax1.axis('off')

    ax2 = plt.subplot(1, 2, 2)
    ax2.set_title('overlay: Pred')
    ax2.imshow(img, cmap='gray')
    for i, cls in enumerate(CLASSES):
        ax2.imshow(pred[cls], alpha=float(alpha), cmap=cmaps[i])
    ax2.axis('off')

    plt.tight_layout()
    return fig_grid, fig_overlay, slice_pos, img_path

case_days_to_vis = {best_case_day, worst_case_day}
gt_rles = load_rles_for_case_days(train_csv, case_days_to_vis)
pred_rles = load_rles_for_case_days(submit_csv, case_days_to_vis)

print('loaded GT case_days:', list(gt_rles.keys()))
print('loaded Pred case_days:', list(pred_rles.keys()))

loaded GT case_days: ['case125_day15', 'case81_day30']
loaded Pred case_days: ['case125_day15', 'case81_day30']


In [8]:
def render_and_save(case_day: str, tag: str):
    if case_day not in case_day_slices:
        raise KeyError(f'case_day not found in inputs/train: {case_day}')
    slice_idx = pick_slice_idx_by_gt_area(case_day, gt_rles.get(case_day, {}))
    slice_pos = slice_pos_map(case_day).get(slice_idx)
    img_path = Path(case_day_slices[case_day][slice_pos])
    img0 = cv2.imread(str(img_path), cv2.IMREAD_GRAYSCALE)
    H, W = img0.shape

    gt = decode_masks(gt_rles.get(case_day, {}).get(slice_idx, {}), H, W)
    pred = decode_masks(pred_rles.get(case_day, {}).get(slice_idx, {}), H, W)

    fig_grid, fig_overlay, slice_pos2, _ = plot_case_day_slice(case_day, slice_idx, gt=gt, pred=pred)

    base = f"vis_{tag}__{case_day}__sliceidx_{slice_idx:04d}_slicepos_{slice_pos2:03d}"
    grid_path = out_dir / f"{base}__grid.png"
    overlay_path = out_dir / f"{base}__overlay.png"
    fig_grid.savefig(grid_path, dpi=150, bbox_inches='tight')
    fig_overlay.savefig(overlay_path, dpi=150, bbox_inches='tight')
    plt.close(fig_grid)
    plt.close(fig_overlay)

    return {
        'case_day': case_day,
        'slice_idx': int(slice_idx),
        'slice_pos': int(slice_pos2),
        'grid_png': grid_path,
        'overlay_png': overlay_path,
    }

best_vis = render_and_save(best_case_day, 'best_cluster')
worst_vis = render_and_save(worst_case_day, 'worst_cluster')

best_vis, worst_vis

({'case_day': 'case125_day15',
  'slice_idx': 98,
  'slice_pos': 97,
  'grid_png': PosixPath('outputs/20260301-002909_Unet3D_IMREAD_UNCHANGED/infer/kmeans_dice_clusters/vis_best_cluster__case125_day15__sliceidx_0098_slicepos_097__grid.png'),
  'overlay_png': PosixPath('outputs/20260301-002909_Unet3D_IMREAD_UNCHANGED/infer/kmeans_dice_clusters/vis_best_cluster__case125_day15__sliceidx_0098_slicepos_097__overlay.png')},
 {'case_day': 'case81_day30',
  'slice_idx': 97,
  'slice_pos': 96,
  'grid_png': PosixPath('outputs/20260301-002909_Unet3D_IMREAD_UNCHANGED/infer/kmeans_dice_clusters/vis_worst_cluster__case81_day30__sliceidx_0097_slicepos_096__grid.png'),
  'overlay_png': PosixPath('outputs/20260301-002909_Unet3D_IMREAD_UNCHANGED/infer/kmeans_dice_clusters/vis_worst_cluster__case81_day30__sliceidx_0097_slicepos_096__overlay.png')})